# Enformer muscle 8-gene paired-track workflow v2 (no predicted-peak panel)

This notebook:
- keeps the same overall workflow structure,
- uses **8 genes** with **2 paired tracks per gene**,
- uses the **modified overlay scripts** so the overlay figures **do not include a predicted-peak panel**,
- writes outputs to a **new run tag** so your earlier figures are not overwritten.

Before running, place these files in your project root:
- `overlay_plotting_no_predpeak_v2.py`
- `run_muscle_report_extension_no_predpeak_v2.py`
- `pred_vs_experiment_encode_fixed_v4.py`


**Important run order**
1. Run Cell 1.
2. Run Cell 2.
3. Restart runtime/session once.
4. Run again from Cell 1 onward.


In [1]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Cell 2 — Rebuild a Colab-compatible Enformer environment
# Run this in a fresh runtime. After it finishes, restart the runtime/session once,
# then run again from Cell 1.

# Remove packages that are commonly left in an incompatible state by older notebook versions.
!pip uninstall -y -q     numpy scipy pandas requests transformers tokenizers huggingface-hub safetensors     enformer-pytorch pyBigWig pyfaidx mygene torchvision torchaudio torchtext || true

# Reinstall the package set that matches the original runnable notebook,
# while also restoring Colab-compatible pandas/requests versions.
!pip install -q --no-cache-dir --force-reinstall     "numpy==1.26.4"     "scipy==1.11.4"     "pandas==2.2.2"     "requests==2.32.4"     "transformers==4.46.3"     "tokenizers==0.20.3"     "huggingface-hub<1.0"     "safetensors>=0.4.3"     "enformer-pytorch"     "pyfaidx"     "mygene"     "pyBigWig"

# These are optional for this project and can trigger CUDA mismatch imports through transformers.
!pip uninstall -y -q torchvision torchaudio torchtext || true

print("Install finished.")
print("Now restart the runtime/session once, then run again from Cell 1.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 300.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 238.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 201.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 167.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 175.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 298.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 298.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 300.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 302.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 283.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 358.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Cell 3 — Version sanity check (run after restarting runtime)
import numpy, scipy, pandas, requests, transformers, torch
import enformer_pytorch

print("numpy =", numpy.__version__)
print("scipy =", scipy.__version__)
print("pandas =", pandas.__version__)
print("requests =", requests.__version__)
print("transformers =", transformers.__version__)
print("torch =", torch.__version__)
print("CUDA available =", torch.cuda.is_available())
print("enformer-pytorch import OK")


numpy = 1.26.4
scipy = 1.11.4
pandas = 2.2.2
requests = 2.32.4
transformers = 4.46.3
torch = 2.11.0+cu130
CUDA available = True
enformer-pytorch import OK


In [4]:
# Cell 4 — Project configuration
from pathlib import Path
from collections import Counter
import os
import sys
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Enformer Final Project").resolve()
DATA_DIR = PROJECT_DIR / "enformer_data"

RUN_TAG = "muscle_8gene_pairs_v2_no_predpeak"

OUTPUT_DIR = PROJECT_DIR / f"outputs_{RUN_TAG}"
REPORT_ROOT = PROJECT_DIR / "report"
REPORT_FIG_DIR = REPORT_ROOT / f"figures_{RUN_TAG}"
REPORT_EXT_DIR = REPORT_ROOT / f"figures_{RUN_TAG}_overlay"
REPORT_PRED_EXP_DIR = REPORT_ROOT / f"figures_{RUN_TAG}_pred_vs_exp"

GENE_LIST_CSV = PROJECT_DIR / "enformer_project_gene_list.csv"
TARGETS_TXT = PROJECT_DIR / "targets_human.txt"
COORDS_CSV = OUTPUT_DIR / "gene_coordinates_hg38.csv"
PAIR_CSV = OUTPUT_DIR / "gene_track_pairs.csv"

CCRE_BED = DATA_DIR / "ccre.bed"
HG38_GZ = DATA_DIR / "hg38.fa.gz"
HG38_FA = DATA_DIR / "hg38-001.fa"

# 8 genes total:
# original 4: ACTA1, MYH7, TNNT1, DES
# new 4 (non-overlapping with the other 4-gene notebook): ACTN2, ADSS1, ASCC1, ASCC3
GENE_TRACK_PAIRS = [
    ("ACTA1", 117), ("ACTA1", 737),
    ("MYH7", 117), ("MYH7", 737),
    ("TNNT1", 117), ("TNNT1", 737),
    ("DES", 117), ("DES", 737),
    ("ACTN2", 117), ("ACTN2", 737),
    ("ADSS1", 117), ("ADSS1", 737),
    ("ASCC1", 116), ("ASCC1", 724),
    ("ASCC3", 116), ("ASCC3", 724),
]

GENES = list(dict.fromkeys(g for g, _ in GENE_TRACK_PAIRS))
UNION_TRACK_INDICES = sorted({int(idx) for _, idx in GENE_TRACK_PAIRS})

# Overlay all 8 genes using one focus track and the modified no-predicted-peak overlay script
OVERLAY_GENES = GENES
FOCUS_TRACK_INDEX = 117   # DNASE: myotube originated from skeletal muscle myoblast

WINDOW_SIZE = 196_608
SAVE_PLOTS = True
RESUME = True
FORCE = False

pair_counts = Counter(g for g, _ in GENE_TRACK_PAIRS)
bad_pair_counts = {g: c for g, c in pair_counts.items() if c != 2}
if bad_pair_counts:
    raise ValueError(f"Each gene should appear exactly twice in GENE_TRACK_PAIRS. Bad counts: {bad_pair_counts}")

if FOCUS_TRACK_INDEX not in UNION_TRACK_INDICES:
    raise ValueError(f"FOCUS_TRACK_INDEX {FOCUS_TRACK_INDEX} is not present in UNION_TRACK_INDICES {UNION_TRACK_INDICES}")

for p in [OUTPUT_DIR, REPORT_ROOT, REPORT_FIG_DIR, REPORT_EXT_DIR, REPORT_PRED_EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

pair_rows = []
display_order_map = {}
for gene, idx in GENE_TRACK_PAIRS:
    display_order = display_order_map.get(gene, 0)
    pair_rows.append({"gene": gene, "track_index": int(idx), "display_order": display_order})
    display_order_map[gene] = display_order + 1

pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(PAIR_CSV, index=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR =", DATA_DIR)
print("RUN_TAG =", RUN_TAG)
print("GENES =", GENES)
print("UNION_TRACK_INDICES =", UNION_TRACK_INDICES)
print("PAIR_CSV =", PAIR_CSV)
display(pair_df)


PROJECT_DIR = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project
DATA_DIR = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_data
RUN_TAG = muscle_8gene_pairs_v2_no_predpeak
GENES = ['ACTA1', 'MYH7', 'TNNT1', 'DES', 'ACTN2', 'ADSS1', 'ASCC1', 'ASCC3']
UNION_TRACK_INDICES = [116, 117, 724, 737]
PAIR_CSV = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/gene_track_pairs.csv


,gene,track_index,display_order
0,ACTA1,117,0
1,ACTA1,737,1
2,MYH7,117,0
3,MYH7,737,1
4,TNNT1,117,0
5,TNNT1,737,1
6,DES,117,0
7,DES,737,1
8,ACTN2,117,0
9,ACTN2,737,1


In [5]:
# Cell 5 — Imports, path checks, and helpers
import gc
import subprocess
import shlex
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

from src.enformer import load_enformer
from src.genes import fetch_gene_coordinates_mygene, load_gene_coordinates, write_gene_coordinates
from src.genome import ensure_uncompressed_fasta, get_fasta_reader, load_gene_symbols
from src.pipeline import run_batch_inference
from src.targets import load_targets
from src.summary import summarize_outputs

def find_existing_file(candidates, label):
    checked = [Path(p) for p in candidates]
    for p in checked:
        if p.exists():
            return p
    checked_str = "\n".join(str(p) for p in checked)
    raise FileNotFoundError(f"Could not find {label}. Checked:\n{checked_str}")

OVERLAY_SCRIPT = find_existing_file([
    PROJECT_DIR / "run_muscle_report_extension_no_predpeak_v2.py",
    PROJECT_DIR / "scripts" / "run_muscle_report_extension_no_predpeak_v2.py",
], "run_muscle_report_extension_no_predpeak_v2.py")

OVERLAY_PLOTTING_SCRIPT = find_existing_file([
    PROJECT_DIR / "overlay_plotting_no_predpeak_v2.py",
    PROJECT_DIR / "scripts" / "overlay_plotting_no_predpeak_v2.py",
], "overlay_plotting_no_predpeak_v2.py")

PRED_EXP_SCRIPT = find_existing_file([
    PROJECT_DIR / "pred_vs_experiment_encode_v5.py",
    PROJECT_DIR / "scripts" / "pred_vs_experiment_encode_v5.py",
], "pred_vs_experiment_encode_v5.py")

required_inputs = [
    PROJECT_DIR,
    DATA_DIR,
    GENE_LIST_CSV,
    TARGETS_TXT,
    CCRE_BED,
    HG38_GZ,
]
missing_inputs = [str(p) for p in required_inputs if not Path(p).exists()]
if missing_inputs:
    raise FileNotFoundError("Required project files/directories are missing:\n" + "\n".join(missing_inputs))

print("Using OVERLAY_SCRIPT =", OVERLAY_SCRIPT)
print("Using OVERLAY_PLOTTING_SCRIPT =", OVERLAY_PLOTTING_SCRIPT)
print("Using PRED_EXP_SCRIPT =", PRED_EXP_SCRIPT)

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_or_fetch_coords(genes, coords_csv: Path):
    coord_map = {}
    missing = list(genes)
    if coords_csv.exists():
        cached = load_gene_coordinates(coords_csv)
        coord_map.update({c.gene: c for c in cached})
        missing = [g for g in genes if g not in coord_map]
    if missing:
        fetched = fetch_gene_coordinates_mygene(missing)
        coord_map.update({c.gene: c for c in fetched})
        write_gene_coordinates(coord_map.values(), coords_csv)
    still_missing = [g for g in genes if g not in coord_map]
    if still_missing:
        raise ValueError(f"Could not fetch coordinates for: {still_missing}")
    return coord_map


Using OVERLAY_SCRIPT = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/run_muscle_report_extension_no_predpeak_v2.py
Using OVERLAY_PLOTTING_SCRIPT = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/overlay_plotting_no_predpeak_v2.py
Using PRED_EXP_SCRIPT = /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/pred_vs_experiment_encode_v5.py


In [6]:
# Cell 6 — Confirm selected union tracks from targets_human.txt
targets_df = load_targets(TARGETS_TXT)
selected_df = targets_df[targets_df["index"].astype(int).isin(UNION_TRACK_INDICES)].copy()

selected_df["track_order"] = selected_df["index"].astype(int).map(
    {idx: i for i, idx in enumerate(UNION_TRACK_INDICES)}
)
selected_df = selected_df.sort_values("track_order").reset_index(drop=True)

found_track_indices = selected_df["index"].astype(int).tolist()
missing_track_indices = [idx for idx in UNION_TRACK_INDICES if idx not in found_track_indices]
if missing_track_indices:
    raise ValueError(
        f"These requested track indices were not found in {TARGETS_TXT.name}: {missing_track_indices}"
    )

track_indices = found_track_indices
track_names = selected_df["description"].astype(str).tolist()

print("Selected track indices:", track_indices)
display(selected_df[["index", "identifier", "description"]])

pd.DataFrame({
    "track_index": track_indices,
    "track_name": track_names,
}).to_csv(OUTPUT_DIR / "track_selection.csv", index=False)

print("Saved:", OUTPUT_DIR / "track_selection.csv")


Selected track indices: [116, 117, 724, 737]


,index,identifier,description
0,116,ENCFF549TYD,DNASE:skeletal muscle myoblast
1,117,ENCFF684FJC,DNASE:myotube originated from skeletal muscle ...
2,724,ENCFF743ROG,CHIP:H3K27ac:skeletal muscle myoblast male adu...
3,737,ENCFF040VCY,CHIP:H3K27ac:myotube


Saved: /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/track_selection.csv


In [7]:
# Cell 7 — Run Enformer prediction for the 8 genes and union of the selected tracks
all_genes_in_csv = load_gene_symbols(GENE_LIST_CSV)
missing_genes = [g for g in GENES if g not in set(all_genes_in_csv)]
if missing_genes:
    raise ValueError(
        f"These genes are not present in {GENE_LIST_CSV.name}: {missing_genes}. "
        "Edit GENE_TRACK_PAIRS in Cell 4 if needed."
    )

coord_map = load_or_fetch_coords(GENES, COORDS_CSV)
print("Fetched coordinates for:", sorted(coord_map.keys()))

fasta_path = ensure_uncompressed_fasta(HG38_GZ, HG38_FA)
fasta = get_fasta_reader(fasta_path)
print("FASTA ready:", fasta_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Prediction device:", device)

model = load_enformer(device=device)

result = run_batch_inference(
    genes=GENES,
    coord_map=coord_map,
    fasta=fasta,
    model=model,
    track_indices=track_indices,
    track_names=track_names,
    output_dir=OUTPUT_DIR,
    window_size=WINDOW_SIZE,
    save_plots=SAVE_PLOTS,
    resume=RESUME,
    force=FORCE,
    plots_dir=REPORT_FIG_DIR,
    device=device,
)

print("Saved:", result.saved)
print("Skipped:", result.skipped)
print("Missing:", result.missing)

cleanup_memory()


Fetched coordinates for: ['ACTA1', 'ACTN2', 'ADSS1', 'ASCC1', 'ASCC3', 'DES', 'MYH7', 'TNNT1']
FASTA ready: /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_data/hg38-001.fa
Prediction device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

Saved: []
Skipped: ['ACTA1', 'MYH7', 'TNNT1', 'DES', 'ACTN2', 'ADSS1', 'ASCC1', 'ASCC3']
Missing: []


In [8]:
# Cell 8 — Summarize outputs
summary_df = summarize_outputs(OUTPUT_DIR, track_selection_csv=OUTPUT_DIR / "track_selection.csv")
display(summary_df.head(30))

print("NPZ files:")
for gene in GENES:
    p = OUTPUT_DIR / f"{gene}_tracks.npz"
    print(gene, "->", p.exists(), p)


,gene,track_index,track_name,mean_signal,max_signal,max_bin,center_signal,bins
0,ACTA1,116,DNASE:skeletal muscle myoblast,0.137905,7.209104,263,4.303153,896
1,ACTA1,117,DNASE:myotube originated from skeletal muscle ...,0.171725,6.693794,263,4.740760,896
2,ACTA1,724,CHIP:H3K27ac:skeletal muscle myoblast male adu...,1.653355,33.735172,260,4.912882,896
3,ACTA1,737,CHIP:H3K27ac:myotube,2.801110,49.956734,260,16.443785,896
4,MYH7,116,DNASE:skeletal muscle myoblast,0.224397,27.431675,708,2.300292,896
5,MYH7,117,DNASE:myotube originated from skeletal muscle ...,0.251083,16.261309,708,2.780722,896
6,MYH7,724,CHIP:H3K27ac:skeletal muscle myoblast male adu...,2.097322,35.712872,443,35.555908,896
7,MYH7,737,CHIP:H3K27ac:myotube,3.719425,49.357224,448,49.357224,896
8,TNNT1,116,DNASE:skeletal muscle myoblast,0.130975,16.016174,201,0.054802,896
9,TNNT1,117,DNASE:myotube originated from skeletal muscle ...,0.181370,13.482887,201,0.147672,896


NPZ files:
ACTA1 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/ACTA1_tracks.npz
MYH7 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/MYH7_tracks.npz
TNNT1 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/TNNT1_tracks.npz
DES -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/DES_tracks.npz
ACTN2 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/ACTN2_tracks.npz
ADSS1 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/ADSS1_tracks.npz
ASCC1 -> True /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/ASCC1_tracks.npz
ASCC3 -> True /content/drive/MyDrive/Colab Noteboo

In [9]:
# Cell 9 — Generate prediction-only waterfall plots directly (no subprocess)
import matplotlib.pyplot as plt
from src.attribution import enformer_bins_to_genome_coords
from src.plotting import plot_tracks_waterfall

outputs_dir = OUTPUT_DIR
out_dir = REPORT_FIG_DIR
out_dir.mkdir(parents=True, exist_ok=True)

windows_path = outputs_dir / "gene_windows.csv"
tracks_path = outputs_dir / "track_selection.csv"
if not windows_path.exists():
    raise FileNotFoundError(f"Missing required file: {windows_path}")
if not tracks_path.exists():
    raise FileNotFoundError(f"Missing required file: {tracks_path}")

windows_df = pd.read_csv(windows_path)
tracks_df = pd.read_csv(tracks_path)

track_map = dict(zip(
    tracks_df["track_index"].astype(int),
    tracks_df["track_name"].astype(str)
))

print("Using outputs_dir:", outputs_dir)
print("Saving figures to:", out_dir)

for gene in GENES:
    npz_path = outputs_dir / f"{gene}_tracks.npz"
    if not npz_path.exists():
        print(f"Skipping {gene}: missing {npz_path.name}")
        continue

    row = windows_df.loc[windows_df["gene"] == gene]
    if row.empty:
        print(f"Skipping {gene}: no row in gene_windows.csv")
        continue
    row = row.iloc[0]

    window_start = int(row["start"])
    tss = int(row["tss"]) if pd.notna(row["tss"]) else None

    npz = np.load(npz_path, allow_pickle=True)
    preds = npz["tracks"]
    local_track_indices = npz["track_indices"].tolist()
    local_track_names = [track_map.get(int(idx), f"track_{idx}") for idx in local_track_indices]

    n_bins = preds.shape[1]
    x_coords = enformer_bins_to_genome_coords(n_bins, window_start)

    print(f"Plotting {gene}")
    print("  preds shape:", preds.shape)
    print("  track_indices:", local_track_indices)

    fig = plot_tracks_waterfall(
        preds,
        track_indices=local_track_indices,
        track_names=local_track_names,
        title=gene,
        normalize="minmax",
        x_offset=None,
        y_scale=1.8,
        y_offset=1.6,
        y_label="Normalized signal (a.u.)",
        right_labels=True,
        x_coords=x_coords,
        tss=tss,
        relative_kb=True,
    )

    save_path = out_dir / f"{gene}_tracks.png"
    fig.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("  saved ->", save_path)

print("Done.")


Using outputs_dir: /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak
Saving figures to: /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak
Plotting ACTA1
  preds shape: (1, 896, 4)
  track_indices: [116, 117, 724, 737]
  saved -> /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/ACTA1_tracks.png
Plotting MYH7
  preds shape: (1, 896, 4)
  track_indices: [116, 117, 724, 737]
  saved -> /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/MYH7_tracks.png
Plotting TNNT1
  preds shape: (1, 896, 4)
  track_indices: [116, 117, 724, 737]
  saved -> /content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/TNNT1_tracks.png
Plotting DES
  preds shape: (1, 896, 4)
  track_indices: [116, 117, 724, 737]
  saved -> /co

In [10]:
# Cell 10 — Generate GI + cCRE overlay figures for all 8 genes (no predicted-peak panel)
env = os.environ.copy()
old_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(PROJECT_DIR) + (f":{old_pythonpath}" if old_pythonpath else "")

cmd = [
    sys.executable,
    str(OVERLAY_SCRIPT),
    "--outputs-dir", str(OUTPUT_DIR),
    "--report-dir", str(REPORT_EXT_DIR),
    "--track-index", str(FOCUS_TRACK_INDEX),
    "--genes", ",".join(OVERLAY_GENES),
    "--ccre-bed", str(CCRE_BED),
    "--gene-list", str(GENE_LIST_CSV),
    "--targets", str(TARGETS_TXT),
    "--coords-csv", str(COORDS_CSV),
    "--fasta-gz", str(HG38_GZ),
    "--fasta-out", str(HG38_FA),
    "--prediction-device", "cuda" if torch.cuda.is_available() else "cpu",
    "--attribution-device", "cpu",
]

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    cwd=str(PROJECT_DIR),
    env=env,
)

print("===== STDOUT =====")
print(result.stdout)
print("===== STDERR =====")
print(result.stderr)
print("===== RETURN CODE =====")
print(result.returncode)

if result.returncode != 0:
    raise RuntimeError("Overlay report script failed.")


Running:
/usr/bin/python3 '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/run_muscle_report_extension_no_predpeak_v2.py' --outputs-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak' --report-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak_overlay' --track-index 117 --genes ACTA1,MYH7,TNNT1,DES,ACTN2,ADSS1,ASCC1,ASCC3 --ccre-bed '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_data/ccre.bed' --gene-list '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_project_gene_list.csv' --targets '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/targets_human.txt' --coords-csv '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/gene_coordinates_hg38.csv' --fasta-gz '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_data/hg38.fa.g

KeyboardInterrupt: 

In [11]:
# Cell 11 — Run prediction-vs-experiment figures using gene-specific paired tracks
env = os.environ.copy()
old_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(PROJECT_DIR) + (f":{old_pythonpath}" if old_pythonpath else "")

cmd = [
    sys.executable,
    str(PRED_EXP_SCRIPT),
    "--project-root", str(PROJECT_DIR),
    "--outputs-dir", str(OUTPUT_DIR),
    "--outdir", str(REPORT_PRED_EXP_DIR),
    "--cache-dir", str(REPORT_ROOT / "encode_bigwig_cache"),
    "--pair-csv", str(PAIR_CSV),
]

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    cwd=str(PROJECT_DIR),
    env=env,
)

print("===== STDOUT =====")
print(result.stdout)
print("===== STDERR =====")
print(result.stderr)
print("===== RETURN CODE =====")
print(result.returncode)

if result.returncode != 0:
    raise RuntimeError("pred_vs_experiment script failed. See STDERR above.")


Running:
/usr/bin/python3 '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/pred_vs_experiment_encode_v5.py' --project-root '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project' --outputs-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak' --outdir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak_pred_vs_exp' --cache-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/encode_bigwig_cache' --pair-csv '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_8gene_pairs_v2_no_predpeak/gene_track_pairs.csv'
===== STDOUT =====
Resolved ENCODE files:
{
  "116": {
    "accession": "ENCFF549TYD",
    "description": "DNASE:skeletal muscle myoblast",
    "download_url": "https://www.encodeproject.org/files/ENCFF549TYD/@@download/ENCFF549TYD.bigWig",
    "local_bigwig": "/content/drive/MyDrive/Colab Notebooks/Enf

In [12]:
# Cell 12 — Preview key output paths
print("Waterfall figures:")
for gene in GENES:
    print(REPORT_FIG_DIR / f"{gene}_tracks.png")

print("\nGI + cCRE overlay figures (no predicted-peak panel):")
for gene in OVERLAY_GENES:
    print(REPORT_EXT_DIR / f"{gene}_overlay.png")

print("\nPrediction-vs-experiment figures:")
for gene in GENES:
    print(REPORT_PRED_EXP_DIR / f"{gene}_pred_vs_exp_matched_layout.png")

print("\nPrediction-only matched-layout figures:")
for gene in GENES:
    print(REPORT_PRED_EXP_DIR / f"{gene}_pred_only_matched_layout.png")


Waterfall figures:
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/ACTA1_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/MYH7_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/TNNT1_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/DES_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/ACTN2_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/ADSS1_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gene_pairs_v2_no_predpeak/ASCC1_tracks.png
/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_8gen

In [14]:
# Cell 13 — Generate NPZ outputs for 100 genes with muscle-matched tracks
from pathlib import Path
import torch

from src.genes import load_gene_coordinates, fetch_gene_coordinates_mygene, write_gene_coordinates
from src.genome import ensure_uncompressed_fasta, get_fasta_reader, load_gene_symbols
from src.enformer import load_enformer
from src.pipeline import run_batch_inference

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Enformer Final Project").resolve()

GENE_LIST_CSV = PROJECT_DIR / "enformer_project_gene_list.csv"
COORDS_CSV_100 = PROJECT_DIR / "outputs_muscle_100genes_v2" / "gene_coordinates_hg38.csv"
OUTPUT_DIR_100 = PROJECT_DIR / "outputs_muscle_100genes_v2"

FASTA_GZ = PROJECT_DIR / "enformer_data" / "hg38.fa.gz"
FASTA_OUT = PROJECT_DIR / "enformer_data" / "hg38-001.fa"

OUTPUT_DIR_100.mkdir(parents=True, exist_ok=True)

TRACK_INDICES_100 = [116, 117, 724, 737, 729, 741]
TRACK_NAMES_100 = [
    "DNASE:skeletal muscle myoblast",
    "DNASE:myotube originated from skeletal muscle myoblast",
    "CHIP:H3K27ac:skeletal muscle myoblast",
    "CHIP:H3K27ac:myotube",
    "CHIP:H3K4me3:skeletal muscle myoblast",
    "CHIP:H3K4me3:myotube",
]

GENES_100 = load_gene_symbols(GENE_LIST_CSV)[:100]
print("Requested genes:", len(GENES_100))

if COORDS_CSV_100.exists():
    cached = load_gene_coordinates(COORDS_CSV_100)
    coord_map_100 = {c.gene: c for c in cached}
else:
    coord_map_100 = {}

missing_100 = [g for g in GENES_100 if g not in coord_map_100]
if missing_100:
    fetched = fetch_gene_coordinates_mygene(missing_100)
    for c in fetched:
        coord_map_100[c.gene] = c
    write_gene_coordinates(coord_map_100.values(), COORDS_CSV_100)

fasta_path_100 = ensure_uncompressed_fasta(FASTA_GZ, FASTA_OUT)
fasta_100 = get_fasta_reader(fasta_path_100)

device_100 = "cuda" if torch.cuda.is_available() else "cpu"
print("Prediction device:", device_100)

model_100 = load_enformer(device=device_100)

result_100 = run_batch_inference(
    genes=GENES_100,
    coord_map=coord_map_100,
    fasta=fasta_100,
    model=model_100,
    track_indices=TRACK_INDICES_100,
    track_names=TRACK_NAMES_100,
    output_dir=OUTPUT_DIR_100,
    window_size=196_608,
    save_plots=False,
    resume=False,
    force=True,
    device=device_100,
)

print("Saved:", result_100.saved[:10], "..." if len(result_100.saved) > 10 else "")
print("Saved count:", len(result_100.saved))
print("Skipped count:", len(result_100.skipped))
print("Missing count:", len(result_100.missing))

Requested genes: 100
Prediction device: cuda
Saved: ['ACTA1', 'ACTN2', 'ADSS1', 'ASCC1', 'ASCC3', 'BIN1', 'CACNA1S', 'CFL2', 'COL12A1', 'COL13A1'] ...
Saved count: 100
Skipped count: 0
Missing count: 0


In [15]:
# Cell 14 — Draw prediction-vs-experiment figures for 100 genes using v5
env = os.environ.copy()
old_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(PROJECT_DIR) + (f":{old_pythonpath}" if old_pythonpath else "")

REPORT_PRED_EXP_DIR_100 = PROJECT_DIR / "report" / "figures_muscle_100genes_pred_vs_exp_v2"
REPORT_PRED_EXP_DIR_100.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(PRED_EXP_SCRIPT),
    "--project-root", str(PROJECT_DIR),
    "--outputs-dir", str(PROJECT_DIR / "outputs_muscle_100genes_v2"),
    "--outdir", str(REPORT_PRED_EXP_DIR_100),
    "--cache-dir", str(PROJECT_DIR / "report" / "encode_bigwig_cache"),
    "--gene-list", str(PROJECT_DIR / "enformer_project_gene_list.csv"),
    "--track-indices", "116,117,724,737,729,741",
]

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    cwd=str(PROJECT_DIR),
    env=env,
)

print("===== STDOUT =====")
print(result.stdout)
print("===== STDERR =====")
print(result.stderr)
print("===== RETURN CODE =====")
print(result.returncode)

if result.returncode != 0:
    raise RuntimeError("pred_vs_experiment v5 script failed. See STDERR above.")

Running:
/usr/bin/python3 '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/pred_vs_experiment_encode_v5.py' --project-root '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project' --outputs-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/outputs_muscle_100genes_v2' --outdir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/figures_muscle_100genes_pred_vs_exp_v2' --cache-dir '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/report/encode_bigwig_cache' --gene-list '/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/enformer_project_gene_list.csv' --track-indices 116,117,724,737,729,741
===== STDOUT =====
Resolved ENCODE files:
{
  "116": {
    "accession": "ENCFF549TYD",
    "description": "DNASE:skeletal muscle myoblast",
    "download_url": "https://www.encodeproject.org/files/ENCFF549TYD/@@download/ENCFF549TYD.bigWig",
    "local_bigwig": "/content/drive/MyDrive/Colab Notebooks/Enformer Final Project/r

In [7]:
#Generating more NPZfile of gene
from pathlib import Path
import torch

from src.genes import load_gene_coordinates, fetch_gene_coordinates_mygene, write_gene_coordinates
from src.genome import ensure_uncompressed_fasta, get_fasta_reader, load_gene_symbols
from src.enformer import load_enformer
from src.pipeline import run_batch_inference

PROJECT_DIR = Path(".")
GENE_LIST_CSV = PROJECT_DIR / "enformer_project_gene_list.csv"
COORDS_CSV = PROJECT_DIR / "gene_coordinates_hg38.csv"
TARGET_OUTPUT_DIR = PROJECT_DIR / "outputs_muscle_100genes"

FASTA_GZ = PROJECT_DIR / "enformer_data" / "hg38.fa.gz"
FASTA_OUT = PROJECT_DIR / "hg38.fa"

# biologically matched muscle tracks
TRACK_INDICES = [116, 117, 724, 737, 729, 741]
TRACK_NAMES = [
    "DNASE:skeletal muscle myoblast",
    "DNASE:myotube originated from skeletal muscle myoblast",
    "CHIP:H3K27ac:skeletal muscle myoblast",
    "CHIP:H3K27ac:myotube",
    "CHIP:H3K4me3:skeletal muscle myoblast",
    "CHIP:H3K4me3:myotube",
]

# 读取 gene list，并截取前 100 个
genes = load_gene_symbols(GENE_LIST_CSV)[:100]
print(f"Requested genes: {len(genes)}")

# 坐标：优先读缓存，没有就在线抓取
if COORDS_CSV.exists():
    cached = load_gene_coordinates(COORDS_CSV)
    coord_map = {c.gene: c for c in cached}
else:
    coord_map = {}

missing = [g for g in genes if g not in coord_map]
if missing:
    fetched = fetch_gene_coordinates_mygene(missing)
    for c in fetched:
        coord_map[c.gene] = c
    write_gene_coordinates(coord_map.values(), COORDS_CSV)

# FASTA
fasta_path = ensure_uncompressed_fasta(FASTA_GZ, FASTA_OUT)
fasta = get_fasta_reader(fasta_path)

# model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_enformer(device=device)

# 批量生成 100 个 genes 的 npz
result = run_batch_inference(
    genes=genes,
    coord_map=coord_map,
    fasta=fasta,
    model=model,
    track_indices=TRACK_INDICES,
    track_names=TRACK_NAMES,
    output_dir=TARGET_OUTPUT_DIR,
    window_size=196_608,
    save_plots=False,
    resume=True,
    force=False,
    device=device,
)

print("Saved:", len(result.saved))
print("Skipped:", len(result.skipped))
print("Missing:", len(result.missing))

Requested genes: 100


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

Saved: 100
Skipped: 0
Missing: 0


In [8]:
#pearson correlation
!PYTHONPATH=. python muscle_biological_correlation.py \
  --project-root . \
  --outputs-dir outputs_muscle_100genes \
  --outdir report/muscle_biological_corr_100genes \
  --gene-list enformer_project_gene_list.csv \
  --tss-window-kb 4

{
  "selected_genes": [
    "ACTA1",
    "ACTN2",
    "ADSS1",
    "ASCC1",
    "ASCC3",
    "BIN1",
    "CACNA1S",
    "CFL2",
    "COL12A1",
    "COL13A1",
    "COL25A1",
    "COL6A1",
    "COL6A2",
    "COL6A3",
    "COX6A2",
    "DNAJB4",
    "DNM2",
    "DOK7",
    "ECEL1",
    "EPG5",
    "FKBP14",
    "FXR1",
    "GBE1",
    "HACD1",
    "HNRNPA2B1",
    "KBTBD13",
    "KLHL40",
    "KLHL41",
    "KY",
    "LETM1",
    "LMNA",
    "LMOD3",
    "MAP3K20",
    "MEGF10",
    "MICU1",
    "MLIP",
    "MTM1",
    "MYBPC1",
    "MYH2",
    "MYH3",
    "MYH7",
    "MYL1",
    "MYL2",
    "MYMK",
    "MYO18B",
    "MYOD1",
    "MYPN",
    "NEB",
    "ORAI1",
    "PAX7",
    "PIEZO2",
    "PYROXD1",
    "RYR1",
    "RYR3",
    "SCN4A",
    "SELENON",
    "SLC25A4",
    "SPEG",
    "SPTBN4",
    "STAC3",
    "STIM1",
    "TNNC2",
    "TNNI2",
    "TNNT1",
    "TNNT3",
    "TPM2",
    "TPM3",
    "TRDN",
    "TRIP4",
    "TTN",
    "UNC45B",
    "VMA21",
    "ZC4H2",
    "CCDC78",
    "CIA